In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# DATASET
class UCFCrimeMultiDataset(Dataset):
    def __init__(self, base_dir):
        self.files = []
        self.labels = []

        categories = sorted(os.listdir(base_dir))
        self.label_map = {cat: i for i, cat in enumerate(categories)}

        print("Classes:", self.label_map)

        for cat in categories:
            cat_path = os.path.join(base_dir, cat)

            if not os.path.isdir(cat_path):
                continue

            for f in os.listdir(cat_path):
                if f.endswith('.npy'):
                    self.files.append(os.path.join(cat_path, f))
                    self.labels.append(self.label_map[cat])

        print(f"Total samples: {len(self.files)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(self.files[idx])  # [segments, 512]
        y = self.labels[idx]

        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# MODEL
class CrimeClassifier(nn.Module):
    def __init__(self, input_dim=512, num_classes=14):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: [batch, segments, 512]

        # Temporal aggregation
        x = torch.mean(x, dim=1)  # [batch, 512]

        return self.net(x)  # [batch, num_classes]

# PATH (YOUR DATASET)
DATA_PATH = "/kaggle/input/datasets/naveenkumar30838/extracted-features/kaggle/working/extracted_features"

# LOAD DATA
dataset = UCFCrimeMultiDataset(DATA_PATH)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# INIT MODEL
num_classes = len(dataset.label_map)

model = CrimeClassifier(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# TRAINING LOOP
EPOCHS = 500

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)   # [B, S, 512]
        y = y.to(device)   # [B]

        optimizer.zero_grad()

        outputs = model(x)  # [B, num_classes]

        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss / len(loader):.4f}")



In [ ]:
# SAVE MODEL
torch.save(model.state_dict(), "/kaggle/working/crime_classifier_mlp.pth")
print("Model saved successfully!")

# SAVE LABEL MAP (IMPORTANT)
np.save("/kaggle/working/label_map_mlp.npy", dataset.label_map)
print("Label map saved!")